# LeetCode 252 — Meeting Rooms bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

- Link đề: https://leetcode.com/problems/meeting-rooms/
- Độ khó: **Easy** · Chủ đề: Array, Sorting, Intervals (khoảng)

Chữ ký hàm LeetCode:

```python
class Solution:
    def canAttendMeetings(self, intervals: List[List[int]]) -> bool:

```

**Mục tiêu học**

1. Hiểu bài toán khoảng (interval): khi nào hai khoảng **chồng nhau (overlap)**.
2. Ôn list lồng nhau, unpacking `start, end = iv`, `sort` với `key=lambda`, `zip`.
3. Baseline so mọi cặp (`O(n²)`), rồi cải tiến: **sắp xếp theo thời điểm bắt đầu, chỉ so hai cuộc họp liền kề** (`O(n log n)`).
4. Chứng minh vì sao chỉ cần so các cặp kề nhau sau khi sắp xếp.
5. Biết cách quét sự kiện (sweep line) — cầu nối sang Meeting Rooms II (253).

## 1. Đề bài và ví dụ

Cho mảng `intervals` với `intervals[i] = [start_i, end_i]` là thời gian một cuộc họp. Hỏi một người có thể **tham dự tất cả** các cuộc họp không (tức là không có hai cuộc họp nào trùng giờ)?

| `intervals` | Kết quả | Giải thích |
|---|---|---|
| `[[0,30],[5,10],[15,20]]` | `False` | `[0,30]` trùm cả `[5,10]` và `[15,20]` |
| `[[7,10],[2,4]]` | `True` | `[2,4]` xong trước khi `[7,10]` bắt đầu |
| `[[1,5],[5,8]]` | `True` | họp 1 kết thúc lúc 5, họp 2 bắt đầu lúc 5 → **không** trùng |
| `[]` | `True` | không có cuộc họp nào |

**Constraints (theo hiểu biết về đề):**

- `0 <= intervals.length <= 10^4`
- `intervals[i].length == 2`
- `0 <= start_i < end_i <= 10^6`

Quy ước quan trọng: khoảng xem như **nửa mở** `[start, end)` — kết thúc lúc 5 và bắt đầu lúc 5 là được (như họp 9h–10h rồi 10h–11h).

## 2. Nền tảng Python

### 2.1. List lồng nhau và unpacking

In [1]:
intervals = [[0, 30], [5, 10], [15, 20]]
print(intervals[1])          # [5, 10]
print(intervals[1][0])       # 5 -> start của cuộc họp thứ 2
for start, end in intervals: # unpacking: mỗi phần tử [s, e] tách thành hai biến
    print(f"họp từ {start} đến {end}, dài {end - start}")

[5, 10]
5
họp từ 0 đến 30, dài 30
họp từ 5 đến 10, dài 5
họp từ 15 đến 20, dài 5


### 2.2. Khi nào hai khoảng chồng nhau?

Hai cuộc họp `[a1, b1)` và `[a2, b2)` **chồng nhau** ⇔ `a1 < b2` **và** `a2 < b1`. Nói cách khác: mỗi cuộc bắt đầu trước khi cuộc kia kết thúc.

Ngược lại, chúng **không** chồng ⇔ `b1 <= a2` (cuộc 1 xong trước) **hoặc** `b2 <= a1` (cuộc 2 xong trước).

In [2]:
def overlap(x, y) -> bool:
    a1, b1 = x
    a2, b2 = y
    return a1 < b2 and a2 < b1

pairs = [([0, 30], [5, 10]), ([2, 4], [7, 10]), ([1, 5], [5, 8]), ([1, 10], [2, 3]), ([5, 8], [1, 5])]
for x, y in pairs:
    print(f"{x} và {y}: chồng nhau? {overlap(x, y)}")

[0, 30] và [5, 10]: chồng nhau? True
[2, 4] và [7, 10]: chồng nhau? False
[1, 5] và [5, 8]: chồng nhau? False
[1, 10] và [2, 3]: chồng nhau? True
[5, 8] và [1, 5]: chồng nhau? False


Dùng `<` (không phải `<=`) để `[1,5]` và `[5,8]` **không** bị coi là chồng nhau.

### 2.3. Sắp xếp list lồng nhau: `sort` và `key`

- `intervals.sort()` sắp tại chỗ; với list lồng nhau, Python so **phần tử đầu trước**, hòa thì so phần tử thứ hai.
- `intervals.sort(key=lambda iv: iv[0])` sắp chỉ theo `start`.
- `sorted(intervals, ...)` trả list **mới**, không đổi list gốc.

In [3]:
data = [[7, 10], [2, 4], [5, 9], [2, 3]]
print(sorted(data))                              # so [start, end] theo thứ tự từ điển
print(sorted(data, key=lambda iv: iv[0]))        # chỉ theo start (giữ thứ tự gốc khi hòa)
print("data gốc không đổi:", data)

[[2, 3], [2, 4], [5, 9], [7, 10]]
[[2, 4], [2, 3], [5, 9], [7, 10]]
data gốc không đổi: [[7, 10], [2, 4], [5, 9], [2, 3]]


`lambda iv: iv[0]` là hàm vô danh: nhận `iv`, trả `iv[0]`. Python dùng giá trị này làm "khóa" để so sánh khi sắp xếp. Sắp xếp của Python là **ổn định (stable)**: phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu.

## 3. Cách 1 — Brute force: kiểm tra mọi cặp

Nếu có **bất kỳ** cặp nào chồng nhau → `False`.

In [4]:
def can_attend_bruteforce(intervals: list[list[int]]) -> bool:
    n = len(intervals)
    for i in range(n):
        for j in range(i + 1, n):       # j > i: mỗi cặp xét đúng một lần
            if overlap(intervals[i], intervals[j]):
                return False
    return True

print(can_attend_bruteforce([[0, 30], [5, 10], [15, 20]]))
print(can_attend_bruteforce([[7, 10], [2, 4]]))
print(can_attend_bruteforce([]))

False
True
True


### Nút thắt

Số cặp = `n(n-1)/2`. Với `n = 10^4` là ~5·10^7 phép so sánh — chậm (có thể Time Limit Exceeded ở Python).

In [5]:
for n in [10, 100, 1000, 10000]:
    print(f"n={n:>5}: số cặp = {n * (n - 1) // 2:,}")

n=   10: số cặp = 45
n=  100: số cặp = 4,950
n= 1000: số cặp = 499,500
n=10000: số cặp = 49,995,000


`{x:,}` định dạng số có dấu phẩy ngăn cách hàng nghìn.

Câu hỏi cải tiến: *Nếu các cuộc họp được xếp theo giờ bắt đầu, có cần so mọi cặp không?*

## 4. Insight — sau khi sắp theo `start`, chỉ cần so các cặp **kề nhau**

Sắp tăng theo `start`. Xét hai cuộc kề nhau `prev` rồi `cur` (`prev.start <= cur.start`). Vì `cur` bắt đầu không sớm hơn `prev`, chúng chồng nhau ⇔ `cur.start < prev.end`.

Vì sao không cần so cặp **không kề**? Giả sử mọi cặp kề đều OK: `end[0] <= start[1]`, `end[1] <= start[2]`, ... Vì `start[i] < end[i]`, ta có chuỗi:

```text
start[0] < end[0] <= start[1] < end[1] <= start[2] < ...
```

Các cuộc họp xếp **nối đuôi** trên trục thời gian, nên cặp không kề càng không thể chồng. Ngược lại, nếu có cặp `(i, j)` chồng nhau thì một cặp kề nào đó giữa `i` và `j` cũng chồng (xem chứng minh ở mục 9).

In [6]:
meetings = [[15, 20], [0, 30], [5, 10]]
meetings_sorted = sorted(meetings, key=lambda iv: iv[0])
print("sau khi sắp:", meetings_sorted)
for prev, cur in zip(meetings_sorted, meetings_sorted[1:]):
    print(f"prev={prev}, cur={cur}: cur.start={cur[0]} < prev.end={prev[1]}? {cur[0] < prev[1]}")

sau khi sắp: [[0, 30], [5, 10], [15, 20]]
prev=[0, 30], cur=[5, 10]: cur.start=5 < prev.end=30? True
prev=[5, 10], cur=[15, 20]: cur.start=15 < prev.end=10? False


`zip(a, a[1:])` ghép mỗi phần tử với phần tử kế tiếp: `(a[0],a[1]), (a[1],a[2]), ...` — mẫu "duyệt cặp kề" rất hay dùng. `a[1:]` là slicing bỏ phần tử đầu.

## 5. Lời giải chính — sắp xếp + so cặp kề

1. Sắp `intervals` theo `start`.
2. Với `i` từ `1` đến `n-1`: nếu `intervals[i][0] < intervals[i-1][1]` → `False`.
3. Hết vòng → `True`.

Mảng rỗng hoặc 1 phần tử: vòng không chạy → `True`.

In [7]:
def can_attend_sort(intervals: list[list[int]]) -> bool:
    ivs = sorted(intervals, key=lambda iv: iv[0])   # không sửa input gốc
    for i in range(1, len(ivs)):
        if ivs[i][0] < ivs[i - 1][1]:
            return False
    return True

for case in [[[0, 30], [5, 10], [15, 20]], [[7, 10], [2, 4]], [[1, 5], [5, 8]], [], [[3, 4]]]:
    print(case, "->", can_attend_sort(case))

[[0, 30], [5, 10], [15, 20]] -> False
[[7, 10], [2, 4]] -> True
[[1, 5], [5, 8]] -> True
[] -> True
[[3, 4]] -> True


## 6. Trace từng bước

`intervals = [[9, 12], [1, 3], [3, 6], [6, 9], [11, 13]]`

Sau khi sắp: `[[1,3], [3,6], [6,9], [9,12], [11,13]]`

| i | prev | cur | `cur.start < prev.end`? | Kết luận |
|---|---|---|---|---|
| 1 | [1,3] | [3,6] | 3 < 3? Không | OK (nối đuôi) |
| 2 | [3,6] | [6,9] | 6 < 6? Không | OK |
| 3 | [6,9] | [9,12] | 9 < 9? Không | OK |
| 4 | [9,12] | [11,13] | 11 < 12? **Có** | chồng → `False` |

In [8]:
def trace_can_attend(intervals):
    ivs = sorted(intervals, key=lambda iv: iv[0])
    print("Sắp theo start:", ivs)
    for i in range(1, len(ivs)):
        prev, cur = ivs[i - 1], ivs[i]
        clash = cur[0] < prev[1]
        print(f"  i={i}: prev={prev}, cur={cur}: {cur[0]} < {prev[1]}? {clash}"
              + ("  -> CHỒNG NHAU, trả False" if clash else "  -> OK"))
        if clash:
            return False
    print("  không có cặp kề nào chồng -> True")
    return True

trace_can_attend([[9, 12], [1, 3], [3, 6], [6, 9], [11, 13]])
print()
trace_can_attend([[7, 10], [2, 4]])

Sắp theo start: [[1, 3], [3, 6], [6, 9], [9, 12], [11, 13]]
  i=1: prev=[1, 3], cur=[3, 6]: 3 < 3? False  -> OK
  i=2: prev=[3, 6], cur=[6, 9]: 6 < 6? False  -> OK
  i=3: prev=[6, 9], cur=[9, 12]: 9 < 9? False  -> OK
  i=4: prev=[9, 12], cur=[11, 13]: 11 < 12? True  -> CHỒNG NHAU, trả False

Sắp theo start: [[2, 4], [7, 10]]
  i=1: prev=[2, 4], cur=[7, 10]: 7 < 4? False  -> OK
  không có cặp kề nào chồng -> True


True

## 7. Cách 3 — Sắp riêng mảng `starts` và `ends`

Một góc nhìn khác: tách `starts = sorted(mọi start)`, `ends = sorted(mọi end)`. Người tham dự được tất cả ⇔ với mọi `i ≥ 1`, cuộc họp thứ `i` (theo giờ bắt đầu) bắt đầu **không sớm hơn** khi cuộc thứ `i-1` kết thúc: `starts[i] >= ends[i-1]`.

In [9]:
def can_attend_starts_ends(intervals):
    starts = sorted(s for s, _ in intervals)
    ends = sorted(e for _, e in intervals)
    return all(starts[i] >= ends[i - 1] for i in range(1, len(intervals)))

for case in [[[0, 30], [5, 10], [15, 20]], [[7, 10], [2, 4]], [[1, 5], [5, 8]], []]:
    print(case, "->", can_attend_starts_ends(case))

[[0, 30], [5, 10], [15, 20]] -> False
[[7, 10], [2, 4]] -> True
[[1, 5], [5, 8]] -> True
[] -> True


Vì sao đúng? Nếu không có chồng nhau, các cuộc họp nối đuôi nên thứ tự theo `start` trùng thứ tự theo `end` → `starts[i]`, `ends[i]` cùng thuộc một cuộc. Nếu có chồng nhau, tại thời điểm chồng có ≥ 2 cuộc đang diễn ra; khi đó tồn tại `i` mà cuộc thứ `i` bắt đầu trước khi `i` cuộc đầu tiên đều kết thúc, tức `starts[i] < ends[i-1]`. Kỹ thuật này là nền của **Meeting Rooms II**.

## 8. Cách 4 — Quét sự kiện (sweep line)

Biến mỗi cuộc họp thành 2 sự kiện: `(start, +1)` và `(end, -1)`. Sắp xếp sự kiện theo thời gian; **khi cùng thời điểm, sự kiện kết thúc (-1) đứng trước** (vì nửa mở). Duyệt, cộng dồn số cuộc đang diễn ra; nếu có lúc > 1 → `False`.

In [10]:
def can_attend_sweep(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))
    events.sort()                 # tuple so phần tử đầu, hòa thì -1 < 1 -> kết thúc trước
    active = 0
    for t, delta in events:
        active += delta
        if active > 1:
            return False
    return True

print(sorted([(5, 1), (5, -1), (1, 1)]))
for case in [[[0, 30], [5, 10], [15, 20]], [[7, 10], [2, 4]], [[1, 5], [5, 8]], []]:
    print(case, "->", can_attend_sweep(case))

[(1, 1), (5, -1), (5, 1)]
[[0, 30], [5, 10], [15, 20]] -> False
[[7, 10], [2, 4]] -> True
[[1, 5], [5, 8]] -> True
[] -> True


Sweep line tổng quát hơn: đổi `active > 1` thành "lấy max của `active`" là ra đáp án Meeting Rooms II (số phòng tối thiểu).

## 9. Code nộp LeetCode

In [11]:
from typing import List

class Solution:
    def canAttendMeetings(self, intervals: List[List[int]]) -> bool:
        intervals.sort(key=lambda iv: iv[0])
        for i in range(1, len(intervals)):
            if intervals[i][0] < intervals[i - 1][1]:
                return False
        return True

sol = Solution()
print(sol.canAttendMeetings([[0, 30], [5, 10], [15, 20]]), sol.canAttendMeetings([[7, 10], [2, 4]]))

False True


Bản nộp dùng `intervals.sort(...)` **tại chỗ** (tiết kiệm bộ nhớ, LeetCode cho phép). Nếu không được phép sửa input, dùng `sorted(...)`.

## 10. Vì sao đúng? (chứng minh)

Sau khi sắp theo `start`: `s[0] <= s[1] <= ... <= s[n-1]`.

**Mệnh đề:** có cặp chồng nhau ⇔ có `i` với `s[i] < e[i-1]`.

- (⇐) Nếu `s[i] < e[i-1]`, và `s[i-1] <= s[i] < e[i]` nên `s[i-1] < e[i]` → hai cuộc kề `i-1`, `i` chồng nhau.
- (⇒) Giả sử mọi cặp kề OK: `e[i-1] <= s[i]` với mọi `i`. Khi đó với `i < j`: `e[i] <= s[i+1] < e[i+1] <= ... <= s[j]` → `e[i] <= s[j]` → cặp `(i, j)` không chồng. Vậy không có cặp nào chồng (phản chứng cho chiều ⇒).

**Invariant vòng lặp:** trước khi xét `i`, các cuộc `0..i-1` đã xếp được nối đuôi không chồng, và `e[i-1]` là thời điểm kết thúc **muộn nhất** trong số đó (vì nối đuôi).

## 11. Độ phức tạp

`n = len(intervals)`.

| Cách | Time | Space phụ |
|---|---|---|
| Brute force mọi cặp | `O(n²)` | `O(1)` |
| **Sắp theo start + so cặp kề** | `O(n log n)` | `O(n)` cho Timsort (`O(1)` thêm ngoài việc sắp) |
| Sắp `starts`/`ends` riêng | `O(n log n)` | `O(n)` |
| Sweep line | `O(n log n)` | `O(n)` (2n sự kiện) |

Phần quét chỉ `O(n)`; chi phí chủ yếu là sắp xếp.

## 12. Bộ test

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

def call(intervals):
    return Solution().canAttendMeetings([iv[:] for iv in intervals])   # copy để test không ảnh hưởng nhau

check("ví dụ 1", call([[0, 30], [5, 10], [15, 20]]), False)
check("ví dụ 2", call([[7, 10], [2, 4]]), True)
check("rỗng", call([]), True)
check("một cuộc", call([[1, 2]]), True)
check("chạm biên (nửa mở)", call([[1, 5], [5, 8]]), True)
check("trùng y hệt", call([[1, 5], [1, 5]]), False)
check("lồng nhau", call([[1, 10], [2, 3]]), False)
check("chồng chỉ ở cặp cuối", call([[9, 12], [1, 3], [3, 6], [6, 9], [11, 13]]), False)
check("cùng start khác end", call([[2, 3], [2, 4]]), False)
check("10^4 cuộc nối đuôi", call([[i, i + 1] for i in range(10000)][::-1]), True)

PASS  ví dụ 1
PASS  ví dụ 2
PASS  rỗng
PASS  một cuộc
PASS  chạm biên (nửa mở)
PASS  trùng y hệt
PASS  lồng nhau
PASS  chồng chỉ ở cặp cuối
PASS  cùng start khác end
PASS  10^4 cuộc nối đuôi


In [13]:
import random
rng = random.Random(252)
solvers = [can_attend_bruteforce, can_attend_sort, can_attend_starts_ends, can_attend_sweep,
           lambda ivs: Solution().canAttendMeetings([iv[:] for iv in ivs])]
stats = {True: 0, False: 0}
for _ in range(3000):
    n = rng.randint(0, 6)
    ivs = []
    for _ in range(n):
        s = rng.randint(0, 20)
        ivs.append([s, s + rng.randint(1, 5)])
    outs = [f(ivs) for f in solvers]
    assert len(set(outs)) == 1, (ivs, outs)
    stats[outs[0]] += 1
print("3000 bộ ngẫu nhiên: 5 cài đặt khớp nhau — PASS", stats)

3000 bộ ngẫu nhiên: 5 cài đặt khớp nhau — PASS {True: 1523, False: 1477}


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` → coi "chạm biên" là chồng nhau

In [14]:
def wrong_le(intervals):
    ivs = sorted(intervals)
    return all(not (ivs[i][0] <= ivs[i - 1][1]) for i in range(1, len(ivs)))

print("[[1,5],[5,8]] ->", wrong_le([[1, 5], [5, 8]]), "(phải True)")

[[1,5],[5,8]] -> False (phải True)


**Vì sao sai:** họp kết thúc lúc 5 thì lúc 5 đã rảnh. **Sửa:** chồng nhau khi `cur.start < prev.end` (dấu `<`).

### Lỗi 2: Quên sắp xếp

In [15]:
def wrong_no_sort(intervals):
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i - 1][1]:
            return False
    return True

print("[[5,10],[0,3]] ->", wrong_no_sort([[5, 10], [0, 3]]), "(phải True)")
print("[[8,9],[1,2],[4,6]] ->", wrong_no_sort([[8, 9], [1, 2], [4, 6]]), "(phải True)")

[[5,10],[0,3]] -> False (phải True)
[[8,9],[1,2],[4,6]] -> False (phải True)


**Vì sao sai:** đề không hứa input đã sắp xếp; so cặp kề chỉ đúng **sau khi** sắp theo `start`. Ở đây các cuộc họp hoàn toàn rời nhau, nhưng vì `[1,2]` đứng sau `[8,9]` nên `1 < 9` bị coi là "chồng". **Sửa:** `intervals.sort(key=lambda iv: iv[0])`.

### Lỗi 3: Sắp theo `end` rồi so như trên

Sắp theo `end` cũng dùng được nếu so đúng chiều (`cur.start < prev.end`)… nhưng người mới hay trộn lẫn hai cách, ví dụ sắp theo `end` mà lại giả định `start` tăng dần. Hãy chọn một quy ước và chứng minh cho nó. Đơn giản nhất: **sắp theo `start`**.

### Lỗi 4: Sweep line đặt sự kiện bắt đầu trước kết thúc khi cùng thời điểm

In [16]:
def wrong_sweep(intervals):
    events = sorted([(s, -1) for s, _ in intervals] + [(e, 1) for _, e in intervals])  # đảo dấu: start=-1 đứng trước
    active = 0
    for _, d in events:
        active += -d          # start làm active tăng
        if active > 1:
            return False
    return True

print("[[1,5],[5,8]] ->", wrong_sweep([[1, 5], [5, 8]]), "(phải True)")

[[1,5],[5,8]] -> False (phải True)


**Vì sao sai:** tại thời điểm 5, sự kiện bắt đầu của cuộc 2 được xử lý trước sự kiện kết thúc của cuộc 1 → `active` tạm thành 2. **Sửa:** khi cùng thời điểm, xử lý **kết thúc trước** (mã hóa `end = -1`, `start = +1` để sắp tự nhiên).

### Lỗi 5: Truy cập `intervals[0]` khi mảng rỗng
Constraints cho phép `intervals.length = 0`. Đừng viết `prev_end = intervals[0][1]` trước khi kiểm tra rỗng.

## 14. Mẹo nhận dạng pattern

- Đề có các **khoảng `[start, end]`** + hỏi chồng nhau / gộp / số phòng → gần như luôn bắt đầu bằng **sắp xếp theo `start`**.
- Sau khi sắp, chỉ cần theo dõi **cuộc vừa xét** (hoặc `end` lớn nhất) — một lần quét.
- Hỏi "tối đa bao nhiêu khoảng cùng lúc" → **sweep line** hoặc **min-heap các `end`**.
- Luôn xác định rõ: biên chạm nhau có tính là chồng không? (`<` hay `<=`).

## 15. Biến thể

- **Merge Intervals (56):** gộp các khoảng chồng nhau.
- **Meeting Rooms II (253):** số phòng họp tối thiểu = số cuộc họp đồng thời lớn nhất.

In [17]:
import heapq

def merge_intervals(intervals):
    res = []
    for s, e in sorted(intervals):
        if res and s <= res[-1][1]:          # Merge Intervals coi chạm biên là gộp
            res[-1][1] = max(res[-1][1], e)
        else:
            res.append([s, e])
    return res

def min_meeting_rooms(intervals):
    ends = []                                 # min-heap các thời điểm kết thúc đang chiếm phòng
    for s, e in sorted(intervals):
        if ends and ends[0] <= s:             # phòng sớm rảnh nhất đã rảnh -> tái dùng
            heapq.heappop(ends)
        heapq.heappush(ends, e)
    return len(ends)

print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))
print(min_meeting_rooms([[0, 30], [5, 10], [15, 20]]), min_meeting_rooms([[7, 10], [2, 4]]))

[[1, 6], [8, 10], [15, 18]]
2 1


`heapq` là **min-heap**: `ends[0]` luôn là phần tử nhỏ nhất; `heappush`/`heappop` `O(log n)`. Lưu ý Meeting Rooms ⇔ `min_meeting_rooms(intervals) <= 1`.

## 16. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay
`[[4, 9], [10, 12], [1, 4], [12, 15], [9, 10]]`: sắp lại rồi kiểm tra từng cặp kề. Kết quả?

### Bài 2 — Hàm `overlap` cho hai khoảng

In [18]:
def exercise_overlap(x: list[int], y: list[int]) -> bool:
    # TODO: chồng nhau khi mỗi cuộc bắt đầu trước khi cuộc kia kết thúc (nửa mở)
    pass

# assert exercise_overlap([1, 5], [4, 8]) == True
# assert exercise_overlap([1, 5], [5, 8]) == False

### Bài 3 — Tự viết `canAttendMeetings`

In [19]:
def exercise_can_attend(intervals: list[list[int]]) -> bool:
    # TODO 1: sắp theo start
    # TODO 2: so cur.start < prev.end
    pass

# assert exercise_can_attend([[0, 30], [5, 10], [15, 20]]) == False
# assert exercise_can_attend([[7, 10], [2, 4]]) == True

### Bài 4 — Trả về cặp cuộc họp xung đột đầu tiên
Sau khi sắp theo start, trả về cặp `(prev, cur)` đầu tiên chồng nhau, hoặc `None`.

In [20]:
def exercise_first_conflict(intervals: list[list[int]]):
    # TODO
    pass

# assert exercise_first_conflict([[0, 30], [5, 10], [15, 20]]) == ([0, 30], [5, 10])
# assert exercise_first_conflict([[7, 10], [2, 4]]) is None

### Bài 5 — Tổng thời gian rảnh giữa các cuộc họp
Giả sử các cuộc họp không chồng nhau. Tính tổng thời gian trống **giữa** cuộc đầu tiên và cuộc cuối cùng. Ví dụ `[[1,3],[6,7],[4,5]]` → `(4-3) + (6-5) = 2`.

In [21]:
def exercise_free_time(intervals: list[list[int]]) -> int:
    # TODO: sắp, cộng (cur.start - prev.end)
    pass

# assert exercise_free_time([[1, 3], [6, 7], [4, 5]]) == 2

### Bài 6 — Mini challenge: Meeting Rooms II bằng sweep line
Trả về số phòng tối thiểu bằng cách quét sự kiện (không dùng heap).

In [22]:
def exercise_min_rooms_sweep(intervals: list[list[int]]) -> int:
    # TODO: events (t, +1/-1), sort, theo dõi max(active)
    pass

# assert exercise_min_rooms_sweep([[0, 30], [5, 10], [15, 20]]) == 2
# assert exercise_min_rooms_sweep([[1, 5], [5, 8]]) == 1

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sắp theo start | `[1,4],[4,9],[9,10],[10,12],[12,15]` | mọi cặp chạm biên → `True` |
| 2 | Mục 2.2 | hai điều kiện nối bằng `and` | `x[0] < y[1] and y[0] < x[1]` |
| 3 | Mục 5 | `sorted(..., key=lambda iv: iv[0])` | dùng `<` |
| 4 | Giống bài 3 | thay `return False` bằng trả cặp | `return (ivs[i-1], ivs[i])` |
| 5 | Khoảng trống giữa hai cuộc kề | `cur[0] - prev[1]` | `sum(...)` qua `zip(ivs, ivs[1:])` |
| 6 | Mục 8 | kết thúc trước bắt đầu khi hòa | `best = max(best, active)` |

## 18. Đáp án tham khảo, template và checklist

Bài 1: sau khi sắp `[[1,4],[4,9],[9,10],[10,12],[12,15]]` — mọi cặp kề chỉ chạm biên → `True`.

In [23]:
def answer_overlap(x, y):
    return x[0] < y[1] and y[0] < x[1]

def answer_can_attend(intervals):
    ivs = sorted(intervals, key=lambda iv: iv[0])
    return all(ivs[i][0] >= ivs[i - 1][1] for i in range(1, len(ivs)))

def answer_first_conflict(intervals):
    ivs = sorted(intervals, key=lambda iv: iv[0])
    for i in range(1, len(ivs)):
        if ivs[i][0] < ivs[i - 1][1]:
            return (ivs[i - 1], ivs[i])
    return None

def answer_free_time(intervals):
    ivs = sorted(intervals)
    return sum(cur[0] - prev[1] for prev, cur in zip(ivs, ivs[1:]))

def answer_min_rooms_sweep(intervals):
    events = sorted([(s, 1) for s, _ in intervals] + [(e, -1) for _, e in intervals])
    active = best = 0
    for _, d in events:
        active += d
        best = max(best, active)
    return best

print(answer_can_attend([[4, 9], [10, 12], [1, 4], [12, 15], [9, 10]]))
assert answer_overlap([1, 5], [4, 8]) and not answer_overlap([1, 5], [5, 8])
assert not answer_can_attend([[0, 30], [5, 10], [15, 20]]) and answer_can_attend([[7, 10], [2, 4]])
assert answer_first_conflict([[0, 30], [5, 10], [15, 20]]) == ([0, 30], [5, 10])
assert answer_first_conflict([[7, 10], [2, 4]]) is None
assert answer_free_time([[1, 3], [6, 7], [4, 5]]) == 2
assert answer_min_rooms_sweep([[0, 30], [5, 10], [15, 20]]) == 2 and answer_min_rooms_sweep([[1, 5], [5, 8]]) == 1
for _ in range(500):
    ivs = [[s, s + rng.randint(1, 4)] for s in (rng.randint(0, 12) for _ in range(rng.randint(0, 6)))]
    assert answer_min_rooms_sweep(ivs) == (min_meeting_rooms(ivs) if ivs else 0)
    assert answer_can_attend(ivs) == (answer_min_rooms_sweep(ivs) <= 1) == can_attend_bruteforce(ivs)
print("Đáp án tham khảo: tất cả test PASS")

True
Đáp án tham khảo: tất cả test PASS


### Template nhớ nhanh

```python
intervals.sort(key=lambda iv: iv[0])
for i in range(1, len(intervals)):
    if intervals[i][0] < intervals[i - 1][1]:   # bắt đầu trước khi cuộc trước kết thúc
        return False
return True
```

### Checklist submit

- [ ] Đã **sắp theo start** trước khi so cặp kề.
- [ ] Dùng `<` (chạm biên `[1,5],[5,8]` là hợp lệ).
- [ ] Mảng rỗng / một phần tử trả `True`, không truy cập `intervals[0]` vô điều kiện.
- [ ] `from typing import List` khi chạy ngoài LeetCode.
- [ ] Nói được vì sao chỉ cần so cặp kề (chuỗi `s[0] < e[0] <= s[1] < e[1] <= ...`).
- [ ] Nêu đúng `O(n log n)` do sắp xếp.